# Mechanistic Interpretability — Full Experiment Notebook
This notebook implements the complete mechanistic interpretability pipeline on Google Colab:
1. Pipeline Setup (distilgpt2 + OpenWebText + SAE training)
2. Quantisation Analysis (8/4/2-bit sweeps with MSE, SDS, CKA, PPL, UMAP)
3. Representation Damage & Causal Importance
4. Mechanistic Explanation & Robust Quantisation


In [ ]:
# ── Mount Google Drive for persistent storage ──
from google.colab import drive
import os, sys, subprocess

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/SAiDL_MI_Data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "samples"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "outputs"), exist_ok=True)
print(f"Persistent storage mounted at {DRIVE_DIR}")

# ── Clone repository ──
REPO_URL = "https://github.com/VvS-2403/SAiDL-Summer-Assignment-2026.git"
REPO_DIR = "/content/SAiDL-Summer-Assignment-2026"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}")

# ── Install dependencies ──
req_path = os.path.join(REPO_DIR, "requirements.txt")
if os.path.exists(req_path):
    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_path], check=True)

# ── Make imports work ──
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")
print("Setup complete.")


In [ ]:
# WandB login
import os
try:
    import wandb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

api_key = os.environ.get("WANDB_API_KEY")
if api_key:
    wandb.login(key=api_key)
    print("WandB logged in via API key.")
else:
    wandb.login()
    print("WandB logged in interactively.")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "distilgpt2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    output_attentions=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(DEVICE)
model.eval()

sample_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "In the future, language models will help scientists discover new ideas.",
    "Causal interventions help identify which parts of a network matter.",
]
inputs = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
print(f"Model loaded on {DEVICE}. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


## Section 1: Activation Patching
Patch a single layer's hidden state tensor and compare logits to understand causal effects.


In [ ]:
def make_replacement_hook(replacement_tensor):
    def hook(module, input_, output):
        if isinstance(output, tuple):
            return (replacement_tensor,) + output[1:]
        return replacement_tensor
    return hook

def fake_quantize(tensor, bits=8):
    """Simulated uniform affine quantization."""
    q_min, q_max = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    t_min, t_max = tensor.min(), tensor.max()
    scale = (t_max - t_min).clamp(min=1e-8) / (q_max - q_min)
    zp = (q_min - (t_min / scale).round()).clamp(q_min, q_max)
    q = (tensor / scale + zp).round().clamp(q_min, q_max)
    return (q - zp) * scale

def patch_layer_and_run(model, tokenizer, text_batch, layer_index, quant_func):
    inp = tokenizer(text_batch, return_tensors="pt", truncation=True, padding=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        clean_out = model(**inp, output_hidden_states=True)
    clean_layer = clean_out.hidden_states[layer_index].detach()
    patched = quant_func(clean_layer)
    handle = model.transformer.h[layer_index - 1].register_forward_hook(
        make_replacement_hook(patched)
    )
    with torch.no_grad():
        out = model(**inp)
    handle.remove()
    return out.logits, clean_layer, patched

logits, clean_h, patched_h = patch_layer_and_run(
    model, tokenizer, sample_texts, layer_index=3,
    quant_func=lambda x: fake_quantize(x, bits=8),
)
print(f"Clean hidden shape: {clean_h.shape}")
print(f"Patched logits shape: {logits.shape}")
print(f"MSE from 8-bit patching: {((clean_h - patched_h)**2).mean().item():.6f}")


## Attention Head Analysis
Visualize attention weights from layer 1, head 1.


In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    attentions = outputs.attentions

print(f"Extracted {len(attentions)} attention layers.")

layer_to_plot, head_to_plot = 0, 0
attn_weights = attentions[layer_to_plot][0, head_to_plot].float().cpu().numpy()

plt.figure(figsize=(6, 5))
plt.imshow(attn_weights, cmap="viridis")
plt.title(f"Layer {layer_to_plot + 1}, Head {head_to_plot + 1} Attention")
plt.xlabel("Key Position")
plt.ylabel("Query Position")
plt.colorbar(label="Attention weight")
plt.tight_layout()
plt.show()


## Feature Extraction & Layer-wise Analysis
Extract hidden states from different layers and visualize structure.


In [ ]:
hidden_states = outputs.hidden_states

for idx, hidden in enumerate(hidden_states):
    print(f"Layer {idx}: shape {hidden.shape}")

# Per-position activation norms at layer 3
layer_acts = hidden_states[3][0].detach().float().cpu().numpy()
mean_activation = np.linalg.norm(layer_acts, axis=-1)

plt.figure(figsize=(8, 3))
plt.plot(mean_activation, marker="o")
plt.title("Activation norm by token position — Layer 3")
plt.xlabel("Token position")
plt.ylabel("Activation norm")
plt.grid(True)
plt.tight_layout()
plt.show()


## Causal Intervention Analysis
Ablate specific neurons and measure the impact on model loss.


In [ ]:
def ablate_neurons(layer_tensor, dims):
    patched = layer_tensor.clone()
    patched[..., dims] = 0.0
    return patched

logits_baseline, _, _ = patch_layer_and_run(model, tokenizer, sample_texts, layer_index=3, quant_func=lambda x: x)
logits_ablate, _, _ = patch_layer_and_run(model, tokenizer, sample_texts, layer_index=3, quant_func=lambda x: ablate_neurons(x, list(range(10))))

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else -100)
inputs_eval = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
labels = inputs_eval.input_ids[:, 1:]

loss_base = loss_fn(logits_baseline[:, :-1, :].reshape(-1, logits_baseline.size(-1)), labels.reshape(-1)).item()
loss_ablate = loss_fn(logits_ablate[:, :-1, :].reshape(-1, logits_ablate.size(-1)), labels.reshape(-1)).item()

print(f"Baseline loss: {loss_base:.4f}")
print(f"Ablated loss (dims 0-9): {loss_ablate:.4f}")
print(f"Delta: {loss_ablate - loss_base:.4f}")


## Layer-wise Representation Similarity
Analyze how representations change across layers.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

layer_norms = [hidden_states[i][0].detach().float().norm(dim=-1).mean().item() for i in range(len(hidden_states))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(layer_norms, marker="o")
ax1.set_title("Mean token representation norm across layers")
ax1.set_xlabel("Layer")
ax1.set_ylabel("Mean norm")
ax1.grid(True)

rep_matrix = np.stack([hidden_states[i][0].mean(dim=0).detach().float().cpu().numpy() for i in range(len(hidden_states))])
similarity = cosine_similarity(rep_matrix)

im = ax2.imshow(similarity, cmap="coolwarm", vmin=0, vmax=1)
ax2.set_title("Layer-wise cosine similarity")
ax2.set_xlabel("Layer")
ax2.set_ylabel("Layer")
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.show()


## Section 1: Extract Activations from OpenWebText
Stream OpenWebText, extract layer-3 hidden states using contiguous 128-token chunks (no padding), normalize, and save to Google Drive.

Set `MAX_TOKENS` to control scale:
- Debug: 200,000 (fast, ~2 min)
- Small: 10,000,000 (10M tokens)
- Full: 80,000,000 (80M tokens, ~1% of corpus)


In [ ]:
import glob
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

SAVE_DIR = Path(DRIVE_DIR) / "samples"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 128
MAX_TOKENS = 200_000  # ← Increase for full run (target: 80_000_000)
FLUSH_EVERY = 500_000
TARGET_LAYER = 3

print(f"Saving activations to Google Drive: {SAVE_DIR}")
print(f"Target tokens: {MAX_TOKENS:,}")

try:
    dataset = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True)
except Exception as exc:
    raise RuntimeError("Failed to load openwebtext.") from exc

def flush_buffer(act_buf, tok_buf, idx):
    stacked_acts = torch.cat(act_buf, dim=0)
    stacked_tokens = torch.cat(tok_buf, dim=0)
    path = SAVE_DIR / f"acts_{idx:04d}.pt"
    torch.save({"acts": stacked_acts.half(), "token_ids": stacked_tokens}, path)
    print(f"\nFlushed {stacked_acts.shape[0]:,} activations to {path}")
    return [], []

activation_buffer = []
token_id_buffer = []
token_buffer = []
total_extracted = 0

# Check how many existing files we have to resume extraction
existing_files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))
if existing_files:
    last_file = existing_files[-1]
    file_idx = int(last_file.split("_")[-1].split(".")[0]) + 1
    total_extracted = file_idx * FLUSH_EVERY
    print(f"Found {len(existing_files)} existing files. Resuming from file index {file_idx}, tokens {total_extracted:,}")
else:
    file_idx = 0

with torch.no_grad():
    for sample in tqdm(dataset, desc="Extracting", initial=total_extracted):
        if total_extracted >= MAX_TOKENS:
            break
            
        text = sample.get("text", "")
        if not text:
            continue
        ids = tokenizer(text, add_special_tokens=False, truncation=True, max_length=model.config.n_ctx)["input_ids"]
        if not ids:
            continue
        token_buffer.extend(ids)

        while len(token_buffer) >= SEQ_LEN:
            chunk = token_buffer[:SEQ_LEN]
            token_buffer = token_buffer[SEQ_LEN:]
            x = torch.tensor([chunk], dtype=torch.long, device=DEVICE)

            outputs = model(x)
            h = outputs.hidden_states[TARGET_LAYER]
            h = h / (h.norm(dim=-1, keepdim=True) + 1e-8)  # Normalize

            activation_buffer.append(h.view(-1, 768).float().cpu())
            token_id_buffer.append(torch.tensor(chunk, dtype=torch.long))
            total_extracted += SEQ_LEN

            if len(activation_buffer) * SEQ_LEN >= FLUSH_EVERY:
                activation_buffer, token_id_buffer = flush_buffer(activation_buffer, token_id_buffer, file_idx)
                file_idx += 1

            if total_extracted >= MAX_TOKENS:
                break

if activation_buffer:
    flush_buffer(activation_buffer, token_id_buffer, file_idx)

print(f"\nExtraction complete. Total tokens: {total_extracted:,}")


## Train Sparse Autoencoder (m=512)
Train a Top-K SAE with bottleneck m=512, sparsity k=51 (10%), Adam lr=1e-4, batch size 4096, for 100k steps.


In [ ]:
import glob
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from mechanistic_interpretability.models.sae import TopKSparseAutoencoder

BOTTLENECK = 512
K = int(0.10 * BOTTLENECK)  # 51
LR = 1e-4
BATCH_SIZE = 4096
TARGET_STEPS = 100_000

sae = TopKSparseAutoencoder(d_model=768, d_sae=BOTTLENECK, k=K).to(DEVICE)
optimizer = optim.Adam(sae.parameters(), lr=LR)

act_files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))
assert act_files, "No activation files found in Drive. Run the extraction cell first."
print(f"Found {len(act_files)} activation files. Training SAE (m={BOTTLENECK}, k={K})...")

sae.train()
global_step = 0

for epoch in range(999):
    for path in act_files:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        activations = (payload["acts"] if isinstance(payload, dict) and "acts" in payload else payload).float()
        ds = TensorDataset(activations)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

        for (x,) in loader:
            x = x.to(DEVICE)
            x_recon, feats, l2_loss = sae(x)
            loss = l2_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
            optimizer.step()
            sae.set_decoder_norm_to_unit_norm()

            global_step += 1
            if global_step % 500 == 0:
                l0 = (feats > 0).float().sum(-1).mean().item()
                var_exp = 1.0 - (l2_loss.item() / x.var().item())
                print(f"step {global_step:>6d} | loss={loss.item():.4f} | l0={l0:.1f} | var_exp={var_exp:.3f}")
            if global_step >= TARGET_STEPS:
                break
        if global_step >= TARGET_STEPS:
            break
    if global_step >= TARGET_STEPS:
        break

SAE_PATH = Path(DRIVE_DIR) / "outputs" / f"sae_m{BOTTLENECK}_k{K}.pt"
torch.save(sae.state_dict(), SAE_PATH)
print(f"\nSaved SAE to {SAE_PATH} after {global_step} steps.")


## Section 2: Quantisation Analysis
Sweep bit-widths (8→4→2) with per-tensor and per-feature quantisation.
Metrics: MSE, SDS, CKA, perplexity (replacement-hook), UMAP visualization.


In [ ]:
import math
import json
import umap
from mechanistic_interpretability.utils.metrics import compute_sds, linear_cka

def quantise(h, bits, mode="per_tensor"):
    q_min, q_max = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    if mode == "per_tensor":
        t_min, t_max = h.min(), h.max()
    else:  # per_feature
        t_min = h.reshape(-1, h.shape[-1]).min(0).values
        t_max = h.reshape(-1, h.shape[-1]).max(0).values
        t_min = t_min.view(*([1] * (h.dim() - 1)), -1)
        t_max = t_max.view(*([1] * (h.dim() - 1)), -1)
    scale = (t_max - t_min).clamp(min=1e-8) / (q_max - q_min)
    zp = (q_min - (t_min / scale).round()).clamp(q_min, q_max)
    q = (h / scale + zp).round().clamp(q_min, q_max)
    return (q - zp) * scale

def compute_ppl_with_quantised_layer(model, tokenizer, text_batch, quant_fn, device, target_layer=3):
    import torch.nn.functional as F
    inp = tokenizer(text_batch, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    input_ids = inp["input_ids"]
    with torch.no_grad():
        out_clean = model(**inp, output_hidden_states=True)
    h_clean = out_clean.hidden_states[target_layer].clone()
    h_quant = quant_fn(h_clean)
    handle = model.transformer.h[target_layer - 1].register_forward_hook(make_replacement_hook(h_quant))
    with torch.no_grad():
        out_quant = model(**inp)
    handle.remove()
    logits = out_quant.logits
    shift_logits = logits[:, :-1, :].reshape(-1, logits.size(-1))
    shift_labels = input_ids[:, 1:].reshape(-1)
    loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=tokenizer.pad_token_id or -100)
    return loss.item(), h_clean, h_quant

# ── Collect ~10k activations for geometry analysis ──
print("Collecting 10k activations for quantisation analysis...")
H_clean_list = []
with torch.no_grad():
    eval_dataset = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True)
    for sample in tqdm(eval_dataset, desc="Collecting", total=300):
        ids = tokenizer(sample["text"], add_special_tokens=False)["input_ids"]
        if len(ids) < 128:
            continue
        chunk = ids[:128]
        x = torch.tensor([chunk], dtype=torch.long, device=DEVICE)
        out = model(x)
        h = out.hidden_states[3]
        h = h / (h.norm(dim=-1, keepdim=True) + 1e-8)
        H_clean_list.append(h.view(-1, 768).float().cpu())
        if len(H_clean_list) * 128 >= 10000:
            break

H_clean_10k = torch.cat(H_clean_list, dim=0)[:10000]
print(f"Collected: {H_clean_10k.shape}")

# ── Sweep ──
BITS_LIST = [8, 4, 2]
QUANT_MODES = ["per_tensor", "per_feature"]
results = {}

for bits in BITS_LIST:
    for mode in QUANT_MODES:
        key = f"{bits}bit_{mode}"
        print(f"\n=== {key} ===")

        H_quant_10k = quantise(H_clean_10k, bits, mode)
        mse = ((H_clean_10k - H_quant_10k) ** 2).mean().item()
        sds = compute_sds(H_clean_10k.numpy(), H_quant_10k.numpy(), k=32)
        cka = linear_cka(H_clean_10k[:1000].numpy(), H_quant_10k[:1000].numpy())

        # PPL via replacement hook
        eval_loss, eval_count = 0.0, 0
        eval_ds = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True)
        for sample in tqdm(eval_ds, desc=f"PPL {key}", total=20):
            loss, _, _ = compute_ppl_with_quantised_layer(
                model, tokenizer, [sample["text"]], lambda h: quantise(h, bits, mode), DEVICE
            )
            eval_loss += loss
            eval_count += 1
            if eval_count >= 20:
                break
        ppl = math.exp(eval_loss / max(eval_count, 1))

        results[key] = {"bits": bits, "mode": mode, "mse": mse, "sds": sds, "cka": cka, "ppl": ppl}
        print(f"{key}: MSE={mse:.6f}, SDS={sds:.4f}, CKA={cka:.4f}, PPL={ppl:.2f}")

        # UMAP on 10k tokens
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
        emb_clean = reducer.fit_transform(H_clean_10k.numpy())
        emb_quant = reducer.transform(H_quant_10k.numpy())

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        ax1.scatter(emb_clean[:, 0], emb_clean[:, 1], s=1, alpha=0.4, c="steelblue")
        ax1.set_title("FP32 Baseline")
        ax1.axis("off")
        ax2.scatter(emb_quant[:, 0], emb_quant[:, 1], s=1, alpha=0.4, c="crimson")
        ax2.set_title(f"{bits}-bit {mode}")
        ax2.axis("off")
        plt.suptitle(f"UMAP: {key}", fontsize=14)
        plt.tight_layout()
        plt.show()

out_path = Path(DRIVE_DIR) / "outputs" / "quant_results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {out_path}")


## Section 3: Representation Damage & Causal Importance
Rank neurons by L2 and KL damage, spectral analysis (SVD spectrum, principal angles, SDS), top-activating tokens, ablation PPL, alignment test.


In [ ]:
import math

TARGET_LAYER = 3

def rank_neurons_by_damage(H_clean, H_quant):
    l2_per_dim = ((H_clean - H_quant) ** 2).mean(0)
    mu_c, sig_c = H_clean.mean(0), H_clean.std(0).clamp(min=1e-8)
    mu_q, sig_q = H_quant.mean(0), H_quant.std(0).clamp(min=1e-8)
    kl_per_dim = (
        (sig_q / sig_c).log()
        + (sig_c ** 2 + (mu_c - mu_q) ** 2) / (2 * sig_q ** 2)
        - 0.5
    )
    return {
        "l2_per_dim": l2_per_dim,
        "kl_per_dim": kl_per_dim,
        "l2_rank": torch.argsort(l2_per_dim, descending=True),
        "kl_rank": torch.argsort(kl_per_dim, descending=True),
    }

def spectral_analysis(H_clean, H_quant, k=64, label="4bit"):
    H_c = (H_clean - H_clean.mean(0)).float()
    H_q = (H_quant - H_quant.mean(0)).float()
    _, S_c, Vc = torch.linalg.svd(H_c, full_matrices=False)
    _, S_q, Vq = torch.linalg.svd(H_q, full_matrices=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(S_c[:200].numpy(), label="FP32", lw=2)
    ax1.plot(S_q[:200].numpy(), label=label, lw=2, ls="--")
    ax1.legend()
    ax1.set_title(f"Singular Value Spectrum: FP32 vs {label}")
    ax1.set_xlabel("Index")
    ax1.set_ylabel("Singular value")

    M = Vc[:k] @ Vq[:k].T
    sv = torch.linalg.svdvals(M).clamp(-1, 1)
    angles_deg = torch.acos(sv) * 180 / torch.pi

    ax2.plot(angles_deg.numpy())
    ax2.set_title(f"Principal Angles: FP32 vs {label}")
    ax2.set_xlabel("Component index")
    ax2.set_ylabel("Angle (degrees)")
    plt.tight_layout()
    plt.show()

    sds = compute_sds(H_clean.numpy(), H_quant.numpy(), k=k)
    print(f"[{label}] SDS={sds:.4f}, mean_angle={angles_deg.mean().item():.2f} deg")
    return angles_deg

def top_activating_tokens_for_dims(dims, n_top=10, max_files=3):
    records = {dim: [] for dim in dims}
    files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))[:max_files]
    for path in files:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        acts = payload["acts"].float()
        token_ids = payload["token_ids"]
        for dim in dims:
            values = acts[:, dim]
            topk = values.topk(min(n_top, len(values)))
            for score, idx in zip(topk.values.tolist(), topk.indices.tolist()):
                tok_id = int(token_ids[idx].item())
                records[dim].append((score, tokenizer.convert_ids_to_tokens([tok_id])[0]))
    for dim in records:
        records[dim] = sorted(records[dim], key=lambda x: x[0], reverse=True)[:n_top]
    return records

def ablation_ppl(model, tokenizer, text_batch, dims_to_zero, device, target_layer=3):
    def zero_hook(module, input_, output):
        h = output[0].clone() if isinstance(output, tuple) else output.clone()
        if dims_to_zero:
            h[:, :, dims_to_zero] = 0.0
        return (h,) + output[1:] if isinstance(output, tuple) else h
    handle = model.transformer.h[target_layer - 1].register_forward_hook(zero_hook)
    inp = tokenizer(text_batch, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    with torch.no_grad():
        out = model(**inp)
    handle.remove()
    ids = inp["input_ids"]
    logits = out.logits
    loss = torch.nn.functional.cross_entropy(
        logits[:, :-1, :].reshape(-1, logits.size(-1)),
        ids[:, 1:].reshape(-1),
        ignore_index=tokenizer.pad_token_id or -100,
    )
    return math.exp(loss.item())

# ── Run analysis ──
H_q4 = quantise(H_clean_10k, 4, "per_tensor")
damage = rank_neurons_by_damage(H_clean_10k, H_q4)
top20_l2 = damage["l2_rank"][:20].tolist()
top20_kl = damage["kl_rank"][:20].tolist()
print("Top-20 L2-damaged dims:", top20_l2)
print("Top-20 KL-damaged dims:", top20_kl)

angles = spectral_analysis(H_clean_10k, H_q4, k=64, label="4bit_per_tensor")

records = top_activating_tokens_for_dims(top20_l2[:5], n_top=5)
print("\nTop activating tokens for top 5 L2-damaged dims:")
for dim, tokens in records.items():
    print(f"  dim {dim}: {tokens}")

# ── Ablation PPL ──
eval_texts = [s["text"] for _, s in zip(range(10), load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True))]
TOP20 = top20_l2[:20]
RAND20 = torch.randperm(768)[:20].tolist()

ppl_baseline = ablation_ppl(model, tokenizer, eval_texts[:5], [], DEVICE)
ppl_top20 = ablation_ppl(model, tokenizer, eval_texts[:5], TOP20, DEVICE)
ppl_rand20 = ablation_ppl(model, tokenizer, eval_texts[:5], RAND20, DEVICE)

print(f"\nPPL baseline: {ppl_baseline:.2f}")
print(f"PPL top-20 L2-damaged ablated: {ppl_top20:.2f}  delta={ppl_top20 - ppl_baseline:.2f}")
print(f"PPL random-20 ablated: {ppl_rand20:.2f}  delta={ppl_rand20 - ppl_baseline:.2f}")

# ── Alignment test ──
def per_dim_ablation_ppl_delta(model, tokenizer, eval_texts, n_dims=50, device=DEVICE):
    baseline = ablation_ppl(model, tokenizer, eval_texts, [], device)
    deltas = {}
    for dim in tqdm(range(n_dims), desc="Per-dim ablation"):
        deltas[dim] = ablation_ppl(model, tokenizer, eval_texts, [dim], device) - baseline
    return deltas

ppl_deltas = per_dim_ablation_ppl_delta(model, tokenizer, eval_texts[:5], n_dims=50)
ppl_rank = sorted(ppl_deltas, key=ppl_deltas.get, reverse=True)
overlap = len(set(TOP20) & set(ppl_rank[:20]))
print(f"\nOverlap: top-20 L2-damaged vs top-20 PPL-critical: {overlap}/20")


## Section 4: Mechanistic Explanation
### 4a. Jacobian Norms — Encoder Sensitivity
Measure how sensitive each SAE bottleneck unit is to input perturbations.


In [ ]:
import torch.nn.functional as F

print("Computing Jacobian norms for SAE encoder sensitivity...")
sae.eval()

# Use a sample batch
sample_payload = torch.load(sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))[0], map_location="cpu", weights_only=False)
sample_acts = sample_payload["acts"].float()[:2048]

N_UNITS_TO_PROBE = min(sae.d_sae, 200)  # Probe up to 200 units
jac_norms = torch.zeros(N_UNITS_TO_PROBE)
n_batches = 0

for start in range(0, len(sample_acts), 256):
    batch = sample_acts[start:start+256].to(DEVICE).requires_grad_(True)
    z = F.relu(sae.encoder(batch - sae.b_dec))  # (B, d_sae)

    for unit_idx in range(N_UNITS_TO_PROBE):
        if batch.grad is not None:
            batch.grad.zero_()
        z_unit_sum = z[:, unit_idx].sum()
        z_unit_sum.backward(retain_graph=(unit_idx < N_UNITS_TO_PROBE - 1))
        jac_norms[unit_idx] += batch.grad.norm(dim=-1).mean().item()

    n_batches += 1

jac_norms /= max(n_batches, 1)

# Plot
jac_sorted, jac_order = jac_norms.sort(descending=True)
plt.figure(figsize=(10, 4))
plt.bar(range(len(jac_sorted)), jac_sorted.numpy(), width=1.0)
plt.title("SAE Encoder: Jacobian Norm per Bottleneck Unit (sorted)")
plt.xlabel("Unit rank")
plt.ylabel("Mean Jacobian norm")
plt.tight_layout()
plt.show()

print(f"Top-10 most sensitive units: {jac_order[:10].tolist()}")
print(f"Top-10 Jacobian norms: {jac_sorted[:10].tolist()}")


### 4b. Fisher-Style Importance Scores
Compute gradient-based importance for each bottleneck unit using the reconstruction loss.


In [ ]:
print("Computing Fisher-style importance scores...")
sae.train()  # need gradients through encoder

fisher_scores = torch.zeros(sae.d_sae)
n_fisher_batches = 0

for start in range(0, len(sample_acts), 256):
    batch = sample_acts[start:start+256].to(DEVICE)
    x_centered = batch - sae.b_dec
    z = F.relu(sae.encoder(x_centered))

    # Reconstruction through top-k
    topk_vals, topk_idx = z.topk(K, dim=-1)
    sparse_z = torch.zeros_like(z)
    sparse_z.scatter_(-1, topk_idx, topk_vals)

    recon = sae.decoder(sparse_z) + sae.b_dec
    loss = F.mse_loss(recon, batch)

    # Per-unit gradient of the loss w.r.t. pre-topk activations
    grad_z = torch.autograd.grad(loss, z, create_graph=False)[0]
    fisher_scores += (grad_z ** 2).mean(0).cpu()
    n_fisher_batches += 1

fisher_scores /= max(n_fisher_batches, 1)

fisher_sorted, fisher_order = fisher_scores.sort(descending=True)
plt.figure(figsize=(10, 4))
plt.semilogy(fisher_sorted.numpy())
plt.title("Fisher Importance Scores per Bottleneck Unit (sorted, log scale)")
plt.xlabel("Unit rank")
plt.ylabel("Fisher score")
plt.tight_layout()
plt.show()

print(f"Top-10 most important units (Fisher): {fisher_order[:10].tolist()}")

# Compare Jacobian vs Fisher rankings
jac_top50 = set(jac_order[:50].tolist())
fisher_top50 = set(fisher_order[:50].tolist())
overlap_jf = len(jac_top50 & fisher_top50)
print(f"Overlap between top-50 Jacobian-sensitive and top-50 Fisher-important: {overlap_jf}/50")

sae.eval()


### 4c. Interaction Tests: Low-Variance Collapse, Sparsity Fragility, Subspace Rotation
Investigate why sparsity and quantisation interact destructively.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# TEST 1: Low-Variance Collapse
# Check if quantised features collapse to fewer distinct values
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 1: Low-Variance Collapse")
print("=" * 60)

for bits in [8, 4, 2]:
    H_q = quantise(H_clean_10k, bits, "per_tensor")
    range_fp = H_clean_10k.max(0).values - H_clean_10k.min(0).values
    range_q = H_q.max(0).values - H_q.min(0).values
    ratio = range_q / (range_fp + 1e-8)
    collapsed = (ratio < 0.5).sum().item()
    print(f"  {bits}-bit: {collapsed}/{H_clean_10k.shape[1]} dims collapsed (range ratio < 0.5), mean ratio={ratio.mean():.4f}")

    if bits == 4:
        plt.figure(figsize=(10, 4))
        plt.hist(ratio.numpy(), bins=50, edgecolor="black", alpha=0.7)
        plt.axvline(0.5, color="red", ls="--", label="Collapse threshold")
        plt.title(f"Range Ratio Distribution ({bits}-bit per_tensor)")
        plt.xlabel("Range(quant) / Range(FP32)")
        plt.ylabel("Count")
        plt.legend()
        plt.tight_layout()
        plt.show()

# ═══════════════════════════════════════════════════════════════
# TEST 2: Sparsity Fragility
# Does quantisation noise change which top-k neurons fire?
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("TEST 2: Sparsity Fragility")
print("=" * 60)

def get_topk_indices(sae_model, activations, k):
    """Get the set of active (top-k) neuron indices for each token."""
    sae_model.eval()
    with torch.no_grad():
        x_centered = activations - sae_model.b_dec
        z = F.relu(sae_model.encoder(x_centered))
        _, topk_idx = z.topk(k, dim=-1)
    return topk_idx

def jaccard_similarity(set_a, set_b):
    """Compute Jaccard similarity between two sets of indices per row."""
    jaccards = []
    for i in range(len(set_a)):
        a = set(set_a[i].tolist())
        b = set(set_b[i].tolist())
        intersection = len(a & b)
        union = len(a | b)
        jaccards.append(intersection / max(union, 1))
    return jaccards

test_batch = sample_acts[:1000].to(DEVICE)
for bits in [8, 4, 2]:
    test_quant = quantise(test_batch, bits, "per_tensor")
    idx_clean = get_topk_indices(sae, test_batch, K).cpu()
    idx_quant = get_topk_indices(sae, test_quant, K).cpu()
    jaccards = jaccard_similarity(idx_clean, idx_quant)
    mean_j = np.mean(jaccards)
    print(f"  {bits}-bit: Jaccard similarity of top-{K} active sets: {mean_j:.4f}")

    if bits == 4:
        plt.figure(figsize=(8, 3))
        plt.hist(jaccards, bins=30, edgecolor="black", alpha=0.7)
        plt.axvline(mean_j, color="red", ls="--", label=f"Mean={mean_j:.3f}")
        plt.title(f"Sparsity Fragility: Jaccard Similarity ({bits}-bit)")
        plt.xlabel("Jaccard similarity")
        plt.ylabel("Count")
        plt.legend()
        plt.tight_layout()
        plt.show()

# ═══════════════════════════════════════════════════════════════
# TEST 3: Subspace Rotation in Bottleneck Space
# How much does the principal subspace of SAE activations rotate?
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("TEST 3: Subspace Rotation")
print("=" * 60)

def sae_encode(sae_model, activations):
    sae_model.eval()
    with torch.no_grad():
        x_centered = activations - sae_model.b_dec
        z = F.relu(sae_model.encoder(x_centered))
        topk_vals, topk_idx = z.topk(K, dim=-1)
        sparse_z = torch.zeros_like(z)
        sparse_z.scatter_(-1, topk_idx, topk_vals)
    return sparse_z

for bits in [8, 4, 2]:
    z_clean = sae_encode(sae, test_batch).cpu().float()
    z_quant = sae_encode(sae, quantise(test_batch, bits, "per_tensor")).cpu().float()

    z_c = z_clean - z_clean.mean(0)
    z_q = z_quant - z_quant.mean(0)
    _, _, Vc = torch.linalg.svd(z_c, full_matrices=False)
    _, _, Vq = torch.linalg.svd(z_q, full_matrices=False)

    k = min(32, Vc.shape[0], Vq.shape[0])
    M = Vc[:k] @ Vq[:k].T
    sv = torch.linalg.svdvals(M).clamp(-1, 1)
    angles = torch.acos(sv) * 180 / torch.pi
    rotation_magnitude = 1.0 - sv.mean().item()
    print(f"  {bits}-bit: mean principal angle = {angles.mean().item():.2f} deg, rotation magnitude = {rotation_magnitude:.4f}")

    if bits == 4:
        plt.figure(figsize=(8, 3))
        plt.plot(angles.numpy())
        plt.title(f"Principal Angles in Bottleneck Space ({bits}-bit)")
        plt.xlabel("Component index")
        plt.ylabel("Angle (degrees)")
        plt.tight_layout()
        plt.show()


### 4d. Subspace-Preserving Quantisation (SPQ)
Decompose activations into important-subspace (8-bit) and residual (2-bit) components.
Compare against standard 4-bit using SDS, CKA, MSE, and **perplexity**.


In [ ]:
class SubspacePreservingQuantiser:
    def __init__(self, H_clean, k=32):
        H_c = (H_clean - H_clean.mean(0)).float()
        _, _, V = torch.linalg.svd(H_c, full_matrices=False)
        self.V_k = V[:k].T
        self.mean = H_clean.mean(0)
        self.k = k

    def _fake_quant(self, x, bits):
        q_min, q_max = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
        t_min, t_max = x.min(), x.max()
        scale = (t_max - t_min).clamp(min=1e-8) / (q_max - q_min)
        zp = (q_min - (t_min / scale).round()).clamp(q_min, q_max)
        q = (x / scale + zp).round().clamp(q_min, q_max)
        return (q - zp) * scale

    def quantise(self, H, bits_important=8, bits_residual=2):
        H_c = (H - self.mean).float()
        V_k = self.V_k.to(H.device)
        coords = H_c @ V_k
        h_imp = coords @ V_k.T
        h_res = H_c - h_imp
        h_imp_q = self._fake_quant(h_imp, bits_important)
        h_res_q = self._fake_quant(h_res, bits_residual)
        return (h_imp_q + h_res_q + self.mean.to(H.device)).to(H.dtype)

spq = SubspacePreservingQuantiser(H_clean_10k[:5000], k=32)
H_spq = spq.quantise(H_clean_10k, bits_important=8, bits_residual=2)
H_std4 = quantise(H_clean_10k, 4, "per_tensor")

# ── Geometry metrics ──
mse_spq = ((H_clean_10k - H_spq) ** 2).mean().item()
mse_std4 = ((H_clean_10k - H_std4) ** 2).mean().item()
sds_spq = compute_sds(H_clean_10k.numpy(), H_spq.numpy(), k=32)
sds_std4 = compute_sds(H_clean_10k.numpy(), H_std4.numpy(), k=32)
cka_spq = linear_cka(H_clean_10k[:1000].numpy(), H_spq[:1000].numpy())
cka_std4 = linear_cka(H_clean_10k[:1000].numpy(), H_std4[:1000].numpy())

# ── PPL comparison ──
print("Computing PPL for standard 4-bit...")
ppl_std4_total, ppl_spq_total, n_eval = 0.0, 0.0, 0
eval_ds = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True)
for sample in tqdm(eval_ds, desc="SPQ PPL", total=20):
    loss_std, _, _ = compute_ppl_with_quantised_layer(
        model, tokenizer, [sample["text"]], lambda h: quantise(h, 4, "per_tensor"), DEVICE
    )
    loss_spq, _, _ = compute_ppl_with_quantised_layer(
        model, tokenizer, [sample["text"]],
        lambda h: spq.quantise(h.cpu(), bits_important=8, bits_residual=2).to(DEVICE), DEVICE
    )
    ppl_std4_total += loss_std
    ppl_spq_total += loss_spq
    n_eval += 1
    if n_eval >= 20:
        break

ppl_std4_val = math.exp(ppl_std4_total / max(n_eval, 1))
ppl_spq_val = math.exp(ppl_spq_total / max(n_eval, 1))

print(f"\n{'Method':30s} {'CKA':>8s} {'MSE':>12s} {'SDS':>8s} {'PPL':>8s}")
print(f"{'Standard 4-bit':30s} {cka_std4:8.4f} {mse_std4:12.6f} {sds_std4:8.4f} {ppl_std4_val:8.2f}")
print(f"{'SPQ (8-bit imp / 2-bit res)':30s} {cka_spq:8.4f} {mse_spq:12.6f} {sds_spq:8.4f} {ppl_spq_val:8.2f}")


## Train SAE (m=1024)
Repeat SAE training with larger bottleneck m=1024, sparsity k=102.


In [ ]:
BOTTLENECK = 1024
K = int(0.10 * BOTTLENECK)  # 102
LR = 1e-4
BATCH_SIZE = 4096
TARGET_STEPS = 100_000

sae_1024 = TopKSparseAutoencoder(d_model=768, d_sae=BOTTLENECK, k=K).to(DEVICE)
optimizer = optim.Adam(sae_1024.parameters(), lr=LR)

act_files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))
assert act_files, "No activation files found."
print(f"Training SAE (m={BOTTLENECK}, k={K})...")

sae_1024.train()
global_step = 0

for epoch in range(999):
    for path in act_files:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        activations = (payload["acts"] if isinstance(payload, dict) and "acts" in payload else payload).float()
        ds = TensorDataset(activations)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

        for (x,) in loader:
            x = x.to(DEVICE)
            x_recon, feats, l2_loss = sae_1024(x)
            loss = l2_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sae_1024.parameters(), 1.0)
            optimizer.step()
            sae_1024.set_decoder_norm_to_unit_norm()

            global_step += 1
            if global_step % 500 == 0:
                l0 = (feats > 0).float().sum(-1).mean().item()
                var_exp = 1.0 - (l2_loss.item() / x.var().item())
                print(f"step {global_step:>6d} | loss={loss.item():.4f} | l0={l0:.1f} | var_exp={var_exp:.3f}")
            if global_step >= TARGET_STEPS:
                break
        if global_step >= TARGET_STEPS:
            break
    if global_step >= TARGET_STEPS:
        break

SAE_PATH_1024 = Path(DRIVE_DIR) / "outputs" / f"sae_m{BOTTLENECK}_k{K}.pt"
torch.save(sae_1024.state_dict(), SAE_PATH_1024)
print(f"\nSaved SAE to {SAE_PATH_1024} after {global_step} steps.")
